## From the feed to the antenna — where the circuit stops being a circuit

Every circuit so far has been small compared with a wavelength, so the same current flowed everywhere in a branch. An antenna is deliberately the opposite: it is made a *significant fraction of a wavelength* long, and the current then varies along it.

$$I(z)=I_0\sin\!\left(k\!\left(\frac{L}{2}-|z|\right)\right),\qquad k=\frac{2\pi}{\lambda}$$

For a half-wave dipole that is a half sine — maximum at the feed, zero at the tips, which it must be since charge has nowhere further to go. Make it a full wavelength and the current at the feed becomes **zero**, so the feedpoint impedance goes enormous; that is why antenna length is not a free parameter.

To the transmitter this looks like an ordinary impedance, $Z_a=R_a+jX_a$, and for a half-wave dipole in free space $R_a\approx73\ \Omega$ with $X_a\approx+42\ \Omega$. Trimming slightly shorter than $\lambda/2$ cancels the reactance, which is why real dipoles are cut to about $0.475\lambda$.

$R_a$ is the strange part. It is a genuine resistance — it dissipates power out of the circuit exactly as a resistor would, and the readout shows the feed current following $P=\frac12I^2R_a$. But nothing gets hot. The power leaves as a wave, and the next section is where it goes.

In [ ]:
FIELDMAP = mpl.colors.LinearSegmentedColormap.from_list("emfield", [
    (0.00, "#eaf3ff"), (0.16, "#5aa9e6"), (0.44, "#0a1622"), (0.50, "#05070b"),
    (0.56, "#211307"), (0.84, "#e08a3c"), (1.00, "#fff3e2")])
ETA0 = 376.730313
C_LIGHT = 2.998e8


def dipole_current(z, Lr):
    """Standing-wave current along a centre-fed dipole, z and L in wavelengths."""
    return np.sin(2 * np.pi * (Lr / 2 - np.abs(z)))


def dipole_Z(Lr):
    """Feedpoint impedance, anchored on the standard half-wave values."""
    Ifeed = np.sin(2 * np.pi * Lr / 2)
    Ra = 73.1 * (Ifeed ** 2) / max(np.sin(np.pi / 2) ** 2, 1e-9)
    Ra = max(Ra, 1e-3) if abs(Ifeed) > 1e-3 else 1e-3
    Xa = 42.5 + 900 * (Lr - 0.5)
    return Ra, Xa


def draw_feed(k, Lr, Vs, f_mhz, match_on):
    lam = C_LIGHT / (f_mhz * 1e6)
    Ra, Xa = dipole_Z(Lr)
    Zs = 50.0
    if match_on:
        Zin = complex(Zs, 0.0)
    else:
        Zin = complex(Ra, Xa)
    I = Vs / (Zs + Zin)
    Pdel = 0.5 * abs(I) ** 2 * Zin.real
    G = abs((Zin - Zs) / (Zin + Zs))
    wt = 2 * np.pi * k / 120
    z = np.linspace(-Lr / 2, Lr / 2, 240)
    Iz = dipole_current(z, Lr) * np.cos(wt)
    vmax = max(Vs, 1e-9)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.8), (-2.6, 2.6))
    wire(a0, [(0.3, 0.35), (2.1, 0.35)], Vs / 2, vmax)
    wire(a0, [(0.3, -0.35), (2.1, -0.35)], -Vs / 2, vmax)
    source(a0, (0.3, -0.35), (0.3, 0.35), 0.0, vmax, "ac", f"{Vs:.1f} V")
    if match_on:
        inductor(a0, (1.0, 0.35), (1.8, 0.35), Vs / 3, vmax, "match")
    resistor(a0, (2.1, 0.35), (2.1, -0.35), 0.0, vmax, None)
    a0.text(2.45, 0.0, f"Ra {Ra:.1f}Ω\nXa {Xa:+.1f}Ω", color=FG, fontsize=7,
            va="center")
    zt = np.linspace(0.06, Lr / 2 * 4, 60)
    for sgn in (1, -1):
        a0.plot([3.6, 3.6], [sgn * 0.08, sgn * Lr / 2 * 4], color=FG, lw=3.0)
    prof = dipole_current(np.linspace(-Lr / 2, Lr / 2, 120), Lr)
    yy = np.linspace(-Lr / 2, Lr / 2, 120) * 4
    a0.fill_betweenx(yy, 3.6, 3.6 + prof * np.cos(wt) * 0.7, color=DOT, alpha=0.45)
    a0.plot(3.6 + prof * np.cos(wt) * 0.7, yy, color=DOT, lw=1.4)
    wire(a0, [(2.1, 0.35), (3.6, 0.08)], Vs / 2, vmax)
    wire(a0, [(2.1, -0.35), (3.6, -0.08)], -Vs / 2, vmax)
    charge_dots(a0, seg((0.3, 0.35), (2.1, 0.35), 30), k / 120 * abs(I) * 60,
                spacing=0.26, ms=3.6)
    charge_dots(a0, seg((2.1, -0.35), (0.3, -0.35), 30), k / 120 * abs(I) * 60,
                spacing=0.26, ms=3.6)
    a0.text(3.6, Lr / 2 * 4 + 0.25, f"{Lr:.3f}λ dipole", color=FG, fontsize=8,
            ha="center")
    a0.set_title("the transmitter, and the current standing wave on the wire")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    for ph, al in ((0, 1.0), (np.pi / 3, 0.35), (2 * np.pi / 3, 0.2)):
        a1.plot(z, dipole_current(z, Lr) * np.cos(wt + ph), color=DOT,
                lw=1.6 if ph == 0 else 0.9, alpha=al)
    a1.plot(z, dipole_current(z, Lr), color=POS, lw=0.9, ls="--",
            label="envelope")
    a1.plot(z, -dipole_current(z, Lr), color=POS, lw=0.9, ls="--")
    a1.axvline(0, color=GRIDC, lw=0.8); a1.axhline(0, color=GRIDC, lw=0.8)
    a1.set_xlabel("position along the dipole  (λ)"); a1.set_ylabel("current")
    a1.legend(fontsize=7)
    a1.set_title(f"feed current {abs(dipole_current(np.array([0.0]), Lr))[0]:.4f} "
                 f"·  tips always zero")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    LL = np.linspace(0.05, 1.05, 400)
    Ras = np.array([dipole_Z(l)[0] for l in LL])
    Xas = np.array([dipole_Z(l)[1] for l in LL])
    a2.plot(LL, Ras, color=POS, lw=1.6, label="Ra")
    a2.plot(LL, Xas, color=ORANGE, lw=1.4, label="Xa")
    a2.axhline(0, color=GRIDC, lw=0.8)
    a2.axvline(Lr, color=FG, lw=1.0, ls="--")
    a2.axvline(0.5, color=DOT, lw=0.9, ls=":")
    a2.set_ylim(-200, 250); a2.set_xlabel("dipole length  (λ)")
    a2.set_ylabel("Ω"); a2.legend(fontsize=7)
    a2.set_title("feedpoint impedance vs length — 73 Ω at a half wave")

    readout(fig, 0.845, 0.90, [
        "TRANSMITTER", "─" * 26,
        f"frequency   {f_mhz:>10.1f}MHz",
        f"λ           {lam:>10.3f}m",
        f"source      {Vs:>10.2f}V",
        f"source Z    {Zs:>10.1f}Ω",
        f"matching    {str(bool(match_on)):>14s}",
        "", "ANTENNA", "─" * 26,
        f"length      {Lr:>10.3f}λ",
        f"physical    {Lr*lam:>10.3f}m",
        f"Ra          {Ra:>10.2f}Ω",
        f"Xa          {Xa:>+10.2f}Ω",
        f"|Γ| to 50Ω  {G:>10.4f}",
        f"VSWR        {(1+G)/max(1-G,1e-9):>10.3f}",
        "", "POWER", "─" * 26,
        f"feed current{abs(I)*1e3:>10.3f}mA",
        f"P = ½I²Ra   {Pdel*1e3:>10.4f}mW",
        f"radiated    {Pdel*1e3:>10.4f}mW",
        f"heat in Ra  {0.0:>10.4f}mW",
        "", "Ra dissipates power",
        "but nothing warms up",
    ], color=GREEN if G < 0.2 else ORANGE)
    footer(fig, f"I(z) = I0 sin(k(L/2−|z|))   ·   Ra ≈ 73 Ω at λ/2   ·   "
                f"P = ½I²Ra leaves as a wave")
    plt.show()


_pA, _sA = timeline(119, step=2)
wA = dict(Lr=widgets.FloatSlider(value=0.5, min=0.05, max=1.0, step=0.005,
                                 description="length (λ):", **SL),
          Vs=widgets.FloatSlider(value=10, min=1, max=30, step=1,
                                 description="source V:", **SL),
          f_mhz=widgets.FloatSlider(value=100, min=10, max=1000, step=10,
                                    description="frequency MHz:", **SL),
          match_on=widgets.Checkbox(value=False, description="matched to 50 Ω",
                                    indent=False),
          k=_sA)
display(widgets.VBox([widgets.HBox([wA["Lr"], wA["Vs"], wA["f_mhz"]]),
                      widgets.HBox([wA["match_on"], _pA, _sA])]),
        widgets.interactive_output(draw_feed, wA))

## The dipole radiating — where the power actually goes

The current on the wire is now the *source term* for a field. Because the field is axially symmetric, the electric field lines are contours of a stream function, with $u=kr$:

$$\Psi(u,\theta,t)=\sin^2\!\theta\left[\underbrace{\frac{\cos(\omega t-u)}{u}}_{\text{bound}}-\underbrace{\vphantom{\frac{1}{u}}\sin(\omega t-u)}_{\text{radiated}}\right]$$

The two terms are equal in size at $u=1$, that is at $r=\lambda/2\pi=0.159\lambda$, and they behave completely differently. Step the phase and watch: loops grow out of the wire, pinch off near that radius, and travel away for ever. Inside it the field is mostly *bound* — it swells and collapses back into the antenna twice per cycle and carries nothing away.

That pinch-off is the radiation, and it is what the 73 Ω is charging for. Select the bound term alone and the loops never detach: energy goes out and comes straight back, which is a reactance, not a resistance.

The right panel is the resulting pattern. A short dipole and a half-wave dipole differ far less than people expect — 1.5 against 1.64 in directivity — because both are essentially $\sin\theta$. What changes hugely with length is the *feedpoint impedance*, not the shape of the radiation.

In [ ]:
def draw_radiate(k, rmax, part, nlev, Lr):
    wt = 2 * np.pi * k / 120
    n = 200
    xg = np.linspace(1e-3, rmax, n)
    zg = np.linspace(-rmax, rmax, n)
    X, Z = np.meshgrid(xg, zg)
    R = np.hypot(X, Z)
    U = 2 * np.pi * R
    s2 = (X / R) ** 2
    bound = s2 * np.cos(wt - U) / U
    rad = -s2 * np.sin(wt - U)
    psi = {"full field": bound + rad, "radiated term only": rad,
           "bound term only": bound}[part]
    psi = np.where(R < 0.055, np.nan, psi)
    pos = np.linspace(0.07, 0.85, nlev)
    tmp = plt.figure()
    cs = tmp.add_subplot(111).contour(X, Z, psi,
                                      levels=np.concatenate([-pos[::-1], pos]))
    segs = [(lv, sg) for lv, sl in zip(cs.levels, cs.allsegs) for sg in sl]
    plt.close(tmp)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.35, 1.0, 0.55], wspace=0.26,
                          left=0.035, right=0.995, top=0.88, bottom=0.12)

    a0 = panel(fig.add_subplot(gs[0]), BLUE)
    for lv, sg in segs:
        a0.plot(sg[:, 0], sg[:, 1], color=POS if lv > 0 else NEG, lw=0.8,
                alpha=0.85)
        a0.plot(-sg[:, 0], sg[:, 1], color=POS if lv > 0 else NEG, lw=0.8,
                alpha=0.85)
    zz = np.linspace(-Lr / 2, Lr / 2, 80)
    a0.plot(np.zeros_like(zz), zz, color=FG, lw=4)
    a0.plot(dipole_current(zz, Lr) * np.cos(wt) * 0.12, zz, color=DOT, lw=1.6)
    th = np.linspace(0, 2 * np.pi, 120)
    rb = 1 / (2 * np.pi)
    a0.plot(rb * np.cos(th), rb * np.sin(th), color=PURP, lw=1.3, ls="--")
    a0.text(rb * 0.75, rb * 1.25, "r = λ/2π", color=PURP, fontsize=7.5)
    a0.set_xlim(-rmax, rmax); a0.set_ylim(-rmax, rmax)
    a0.set_aspect("equal"); a0.grid(False)
    a0.set_xlabel("x  (λ)"); a0.set_ylabel("z  (λ)")
    a0.set_title(f"electric field lines — ωt = {np.degrees(wt) % 360:.0f}°  ({part})")

    a1 = panel(fig.add_subplot(gs[1], projection="polar"), ORANGE)
    a1.set_facecolor(PANEL)
    th2 = np.linspace(0, 2 * np.pi, 721)
    st = np.abs(np.sin(th2))
    kl = np.pi * Lr
    with np.errstate(divide="ignore", invalid="ignore"):
        F = np.abs((np.cos(kl * np.cos(th2)) - np.cos(kl)) / np.where(st < 1e-6, 1, st))
    F = np.where(st < 1e-6, 0, F)
    F = F / max(F.max(), 1e-9)
    a1.plot(th2, F, color=ORANGE, lw=1.6)
    a1.fill_between(th2, 0, F, color=ORANGE, alpha=0.2)
    a1.plot(th2, st / st.max(), color=MUTED, lw=0.9, ls="--")
    a1.set_theta_zero_location("N")
    a1.set_ylim(0, 1.05); a1.tick_params(colors=MUTED, labelsize=6.5)
    a1.grid(alpha=0.2, color=GRIDC)
    a1.set_title("pattern — dashed is a short dipole (sinθ)", pad=14)

    Dhw = 1.64
    readout(fig, 0.845, 0.88, [
        "FIELD REGIONS", "─" * 26,
        f"reactive    r < {1/(2*np.pi):.4f}λ",
        f"crossover   u = kr = 1",
        f"shown out to{rmax:>10.2f}λ",
        f"phase       {np.degrees(wt)%360:>10.0f}°",
        "", "TERMS", "─" * 26,
        "bound     ~ 1/u",
        "radiated  ~ 1",
        "equal at r = λ/2π",
        "", "PATTERN", "─" * 26,
        f"length      {Lr:>10.3f}λ",
        f"directivity {(1.5 if Lr<0.2 else Dhw):>10.2f}",
        f"in dBi      {10*np.log10(1.5 if Lr<0.2 else Dhw):>10.2f}",
        "", "short dipole 1.50",
        "half wave    1.64",
        "the shape barely",
        "changes — the",
        "impedance changes",
        "enormously",
    ])
    footer(fig, "Ψ = sin²θ[cos(ωt−u)/u − sin(ωt−u)]   ·   "
                "loops pinch off at r ≈ λ/2π and never come back")
    plt.show()


_pB, _sB = timeline(119, step=2)
wB = dict(rmax=widgets.FloatSlider(value=2.0, min=0.8, max=4.0, step=0.2,
                                   description="extent (λ):", **SL),
          part=widgets.Dropdown(options=["full field", "radiated term only",
                                         "bound term only"],
                                value="full field", description="show:", **SL),
          nlev=widgets.IntSlider(value=5, min=3, max=9, step=1,
                                 description="line density:", **SL),
          Lr=widgets.FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05,
                                 description="length (λ):", **SL),
          k=_sB)
display(widgets.VBox([widgets.HBox([wB["rmax"], wB["part"], wB["nlev"]]),
                      widgets.HBox([wB["Lr"], _pB, _sB])]),
        widgets.interactive_output(draw_radiate, wB))

## The crossing — one wave, two antennas, and a delay

Put a second dipole some distance away and the field takes time to reach it. The picture below is the instantaneous $E_z$ of the wave leaving the transmitter, and the receiving element only starts producing anything after

$$t_d=\frac{R}{c}$$

which for the default 10 m at 100 MHz is 33.4 ns, a third of a period. The trace under the field shows the transmit current and the induced voltage on the same axis, and the gap between them opening as you move the receiver away is the propagation delay — the same thing radar measures for range.

Two things are worth watching. Move the receiver out and the amplitude falls as $1/R$ — the *field* falls as $1/R$, and power therefore as $1/R^2$, which is the one-way spreading loss from the EW notebook arriving here from the circuit side.

Then rotate the receiving dipole. At 90° to the transmitter it picks up nothing at all, because the induced voltage depends on the component of $\mathbf{E}$ along the wire:

$$V_{oc}=\mathbf{E}\cdot\mathbf{h}_{eff}=E\,h_{eff}\cos\psi$$

Cross-polarisation is not attenuation, it is a dot product going to zero — which is why polarisation is a resource that can be reused, and why a misaligned antenna can be deaf to a very strong signal.

In [ ]:
def draw_link(k, R_m, f_mhz, tilt_deg, Pt_W):
    lam = C_LIGHT / (f_mhz * 1e6)
    Rl = R_m / lam
    wt = 2 * np.pi * k / 140
    G = 1.64
    Ra = 73.1
    heff = lam / np.pi
    E0 = np.sqrt(2 * ETA0 * Pt_W * G / (4 * np.pi * R_m ** 2))
    Voc = E0 * heff * np.cos(np.deg2rad(tilt_deg))
    td = R_m / C_LIGHT

    ext = max(Rl * 1.35, 2.0)
    n = 220
    xs = np.linspace(-0.35 * ext, ext, n)
    zs = np.linspace(-ext * 0.62, ext * 0.62, n)
    X, Zc = np.meshgrid(xs, zs)
    Rg = np.hypot(X, Zc)
    st2 = np.where(Rg > 1e-6, (X / np.maximum(Rg, 1e-9)) ** 2, 0.0)
    Ez = np.where(Rg > 0.08,
                  np.sqrt(st2) * np.cos(wt - 2 * np.pi * Rg) / np.maximum(Rg, 0.08),
                  0.0)
    sc = np.percentile(np.abs(Ez), 99) or 1.0

    fig = plt.figure(figsize=(13.0, 5.4))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.5, 1.0, 0.55],
                          height_ratios=[1.25, 1], wspace=0.26, hspace=0.42,
                          left=0.035, right=0.995, top=0.90, bottom=0.10)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.imshow(Ez, extent=[xs[0], xs[-1], zs[0], zs[-1]], origin="lower",
              cmap=FIELDMAP, vmin=-sc, vmax=sc, aspect="equal",
              interpolation="bilinear")
    zz = np.linspace(-0.25, 0.25, 40)
    a0.plot(np.zeros_like(zz), zz, color="#f2f5fa", lw=4)
    a0.plot(dipole_current(zz, 0.5) * np.cos(wt) * 0.18, zz, color=DOT, lw=1.6)
    tl = np.deg2rad(tilt_deg)
    a0.plot([Rl - 0.25 * np.sin(tl), Rl + 0.25 * np.sin(tl)],
            [-0.25 * np.cos(tl), 0.25 * np.cos(tl)], color=GREEN, lw=4)
    a0.plot([0, Rl], [0, 0], color=CYAN if False else PURP, lw=0.8, ls=":")
    a0.text(0, -0.42, "TX", color="#f2f5fa", fontsize=8, ha="center")
    a0.text(Rl, -0.42, "RX", color=GREEN, fontsize=8, ha="center")
    a0.grid(False)
    a0.set_xlabel("distance  (λ)"); a0.set_ylabel("(λ)")
    a0.set_title(f"instantaneous field — {R_m:.1f} m = {Rl:.2f}λ apart, "
                 f"delay {td*1e9:.1f} ns")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    t = np.linspace(0, 3 / (f_mhz * 1e6), 700)
    itx = np.cos(2 * np.pi * f_mhz * 1e6 * t)
    vrx = np.where(t > td, np.cos(2 * np.pi * f_mhz * 1e6 * (t - td)), 0.0)
    a1.plot(t * 1e9, itx, color=DOT, lw=1.4, label="TX current")
    a1.plot(t * 1e9, vrx, color=GREEN, lw=1.4, label="RX voltage")
    a1.axvline(td * 1e9, color=PURP, lw=1.0, ls="--")
    a1.text(td * 1e9, 1.12, f" {td*1e9:.1f} ns", color=PURP, fontsize=7)
    a1.axvline(k / 140 * 3 / (f_mhz * 1e6) * 1e9 % (t[-1] * 1e9), color=FG,
               lw=0.9, ls=":")
    a1.set_ylim(-1.3, 1.35); a1.set_xlabel("time  (ns)")
    a1.legend(fontsize=7); a1.set_title("the delay is the distance")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    tt = np.linspace(0, 90, 200)
    a2.plot(tt, np.cos(np.deg2rad(tt)) ** 2, color=GREEN, lw=1.6)
    a2.axvline(tilt_deg, color=FG, lw=1.0, ls="--")
    a2.plot([tilt_deg], [np.cos(tl) ** 2], "o", ms=7, color=DOT)
    a2.set_xlabel("polarisation mismatch  (deg)")
    a2.set_ylabel("power fraction")
    a2.set_title(f"cos²ψ — {100*np.cos(tl)**2:.1f}% of the power gets in")

    readout(fig, 0.845, 0.90, [
        "LINK", "─" * 26,
        f"frequency   {f_mhz:>10.1f}MHz",
        f"λ           {lam:>10.3f}m",
        f"distance    {R_m:>10.2f}m",
        f"in wavel.   {Rl:>10.3f}λ",
        f"delay R/c   {td*1e9:>10.2f}ns",
        f"in cycles   {td*f_mhz*1e6:>10.3f}",
        "", "FIELD AT RX", "─" * 26,
        f"Pt          {Pt_W:>10.2f}W",
        f"|E|         {E0*1e3:>10.4f}mV/m",
        f"S = E²/2η   {E0**2/(2*ETA0)*1e6:>10.4f}µW/m²",
        "", "INDUCED", "─" * 26,
        f"h_eff = λ/π {heff:>10.4f}m",
        f"tilt ψ      {tilt_deg:>10.1f}°",
        f"cos ψ       {np.cos(tl):>10.4f}",
        f"Voc         {Voc*1e3:>10.4f}mV",
        "", f"E falls as 1/R",
        f"power as 1/R²",
        "cross-pol is a dot",
        "product, not a loss",
    ], color=NEG if abs(tilt_deg) > 75 else FG)
    footer(fig, f"Voc = E·h_eff·cosψ   ·   h_eff = λ/π for a half wave   ·   "
                f"delay = R/c = {td*1e9:.2f} ns")
    plt.show()


_pC, _sC = timeline(139, step=2)
wC = dict(R_m=widgets.FloatSlider(value=10, min=2, max=40, step=0.5,
                                  description="distance (m):", **SL),
          f_mhz=widgets.FloatSlider(value=100, min=50, max=300, step=10,
                                    description="frequency MHz:", **SL),
          tilt_deg=widgets.FloatSlider(value=0, min=0, max=90, step=5,
                                       description="RX tilt (°):", **SL),
          Pt_W=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="Pt (W):", **SL),
          k=_sC)
display(widgets.VBox([widgets.HBox([wC["R_m"], wC["f_mhz"], wC["tilt_deg"]]),
                      widgets.HBox([wC["Pt_W"], _pC, _sC])]),
        widgets.interactive_output(draw_link, wC))

## The receiving antenna — a Thévenin source you cannot see

The wave arrives, drives current along the receiving wire, and from the receiver's point of view the antenna is nothing more exotic than a source with an internal impedance — Thévenin, exactly as in the theory notebook:

$$V_{oc}=E\,h_{eff},\qquad Z_a=R_a+jX_a$$

So the whole of maximum power transfer applies unchanged. The load should be the **conjugate** $R_a-jX_a$, and then

$$P_{load}=\frac{|V_{oc}|^2}{8R_a}$$

Here is the check that ties the two halves of this series together. That expression is pure circuit theory — a voltage source, an internal resistance, a matched load. The field-side answer is $P=S\cdot A_e$ with $S=E^2/2\eta$ and $A_e=G\lambda^2/4\pi$, and it contains no circuit quantities at all. Computed both ways in the readout they agree to **0.03%**, the residual being the rounding in $G=1.64$ and $R_a=73.1$.

Half the available power is unavoidably lost even at a perfect match, exactly as in the theory notebook — the antenna re-radiates it. And the mismatch curve is the same shape as before: drag the load away from $R_a$ in either direction and the delivered power falls symmetrically.

The rest is the receiver you already built: match, filter, mix, detect. The antenna was never a different kind of object.

In [ ]:
def draw_receiver(k, E_mV, f_mhz, RL, XL, detect_on):
    lam = C_LIGHT / (f_mhz * 1e6)
    Ra, Xa = 73.1, 42.5
    G, E = 1.64, E_mV * 1e-3
    heff = lam / np.pi
    Voc = E * heff
    Zl = complex(RL, XL)
    I = Voc / (complex(Ra, Xa) + Zl)
    Pl = 0.5 * abs(I) ** 2 * RL
    Pmax = Voc ** 2 / (8 * Ra)
    S = E ** 2 / (2 * ETA0)
    Ae = G * lam ** 2 / (4 * np.pi)
    Pem = S * Ae
    wt = 2 * np.pi * k / 120
    vmax = max(Voc, 1e-12)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.2, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.6, 4.8), (-1.6, 1.8))
    zz = np.linspace(-1.0, 1.0, 60)
    a0.plot(np.zeros_like(zz) + 0.5, zz, color=GREEN, lw=4)
    a0.plot(0.5 + dipole_current(zz / 4, 0.5) * np.cos(wt) * 0.35, zz,
            color=DOT, lw=1.4)
    wire(a0, [(0.5, 0.12), (1.6, 0.35)], Voc / 2, vmax)
    wire(a0, [(0.5, -0.12), (1.6, -0.35)], -Voc / 2, vmax)
    resistor(a0, (1.6, 0.35), (3.0, 0.35), Voc / 2, vmax, f"RL {RL:.0f}Ω")
    if XL >= 0:
        inductor(a0, (3.0, 0.35), (4.0, 0.35), 0.0, vmax, f"+j{XL:.0f}")
    else:
        capacitor(a0, (3.0, 0.35), (4.0, 0.35), 0.0, vmax, f"−j{abs(XL):.0f}")
    wire(a0, [(4.0, 0.35), (4.4, 0.35), (4.4, -0.35), (1.6, -0.35)],
         -Voc / 2, vmax)
    charge_dots(a0, seg((1.6, 0.35), (3.0, 0.35), 30), k / 120 * abs(I) * 3e5,
                spacing=0.24, ms=3.4)
    a0.set_title("the wave drives the wire, the wire drives the load",
                 fontsize=8.5)

    a1 = fig.add_subplot(gs[1, 0])
    sch_axes(a1, (-0.6, 4.8), (-1.0, 1.4))
    source(a1, (0.6, -0.5), (0.6, 0.7), 0.0, vmax, "ac", f"Voc {Voc*1e3:.3f}mV")
    resistor(a1, (0.6, 0.7), (1.9, 0.7), Voc / 2, vmax, f"Ra {Ra:.0f}Ω")
    inductor(a1, (1.9, 0.7), (2.9, 0.7), Voc / 3, vmax, f"Xa +{Xa:.0f}")
    resistor(a1, (2.9, 0.7), (4.2, 0.7), Voc / 4, vmax, f"load")
    wire(a1, [(4.2, 0.7), (4.5, 0.7), (4.5, -0.5), (0.6, -0.5)], 0.0, vmax)
    charge_dots(a1, loop_rect(0.6, 4.5, -0.5, 0.7), k / 120 * abs(I) * 3e5,
                spacing=0.24, ms=3.4)
    a1.set_title("the same thing as a Thévenin source", fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    rr = np.logspace(0, 4, 400)
    P = np.array([0.5 * abs(Voc / (complex(Ra, Xa) + complex(r, XL))) ** 2 * r
                  for r in rr])
    a2.semilogx(rr, P * 1e12, color=POS, lw=1.7)
    a2.axvline(Ra, color=DOT, lw=1.1, ls="--")
    a2.text(Ra * 1.2, P.max() * 1e12 * 0.5, "RL = Ra", color=DOT, fontsize=7.5)
    a2.plot([RL], [Pl * 1e12], "o", ms=8, color=ORANGE)
    a2.set_xlabel("load resistance  (Ω)"); a2.set_ylabel("power  (pW)")
    a2.set_title(f"peak {Pmax*1e12:.4f} pW at the conjugate match")

    a3 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a3.bar([0, 1], [Pmax * 1e12, Pem * 1e12], color=[POS, PURP], width=0.5)
    for x, v in ((0, Pmax * 1e12), (1, Pem * 1e12)):
        a3.text(x, v, f"  {v:.5f} pW", ha="center", va="bottom", fontsize=8)
    a3.set_xticks([0, 1])
    a3.set_xticklabels(["circuit:\n$V_{oc}^2/8R_a$", "field:\n$S\\cdot A_e$"],
                       fontsize=8)
    a3.set_ylabel("available power  (pW)")
    a3.set_ylim(0, max(Pmax, Pem) * 1e12 * 1.35)
    a3.set_title(f"two independent routes — agree to "
                 f"{abs(Pmax-Pem)/Pem*100:.3f}%")

    readout(fig, 0.845, 0.90, [
        "INCOMING", "─" * 26,
        f"|E|         {E_mV:>10.3f}mV/m",
        f"frequency   {f_mhz:>10.1f}MHz",
        f"λ           {lam:>10.3f}m",
        f"S = E²/2η   {S*1e9:>10.4f}nW/m²",
        "", "ANTENNA", "─" * 26,
        f"h_eff = λ/π {heff:>10.4f}m",
        f"Voc         {Voc*1e3:>10.5f}mV",
        f"Ra          {Ra:>10.1f}Ω",
        f"Xa          {Xa:>+10.1f}Ω",
        f"Ae = Gλ²/4π {Ae:>10.5f}m²",
        "", "LOAD", "─" * 26,
        f"RL          {RL:>10.1f}Ω",
        f"XL          {XL:>+10.1f}Ω",
        f"conjugate?  {str(abs(RL-Ra)<1 and abs(XL+Xa)<1):>14s}",
        f"|I|         {abs(I)*1e6:>10.4f}µA",
        f"P delivered {Pl*1e12:>10.5f}pW",
        "", "AVAILABLE", "─" * 26,
        f"Voc²/8Ra    {Pmax*1e12:>10.5f}pW",
        f"S·Ae        {Pem*1e12:>10.5f}pW",
        f"difference  {abs(Pmax-Pem)/Pem*100:>10.3f}%",
    ], color=GREEN if abs(RL - Ra) < 1 and abs(XL + Xa) < 1 else FG)
    footer(fig, f"Voc = E·λ/π   ·   P = Voc²/8Ra = S·Ae   ·   "
                f"circuit theory and field theory give the same number")
    plt.show()


_pD, _sD = timeline(119, step=2)
wD = dict(E_mV=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="|E| (mV/m):", **SL),
          f_mhz=widgets.FloatSlider(value=100, min=50, max=300, step=10,
                                    description="frequency MHz:", **SL),
          RL=widgets.FloatSlider(value=73, min=5, max=600, step=1,
                                 description="load R (Ω):", **SL),
          XL=widgets.FloatSlider(value=-42.5, min=-200, max=200, step=2.5,
                                 description="load X (Ω):", **SL),
          detect_on=widgets.Checkbox(value=False, description="(reserved)",
                                     indent=False),
          k=_sD)
display(widgets.VBox([widgets.HBox([wD["E_mV"], wD["f_mhz"], wD["RL"]]),
                      widgets.HBox([wD["XL"], _pD, _sD])]),
        widgets.interactive_output(draw_receiver, wD))